# Text: table to prediction

The same five steps as the image pipeline, on the modality where the data
usually starts as a spreadsheet. `upload()` reads the table, the prep agent maps
its columns, and everything after that is identical — which is the point of one
dataset abstraction across modalities.

In [ ]:
import orinth
from pathlib import Path

# A .csv or .jsonl with a text column and a label column.
SOURCE = Path("~/Downloads/my-text.csv").expanduser()
print(SOURCE, "exists:", SOURCE.exists())

In [ ]:
import csv
import tempfile

if not SOURCE.exists():
    SOURCE = Path(tempfile.mkdtemp()) / "example-support-tickets.csv"
    ROWS = [
        ("billing", "I was charged twice for the same month"),
        ("billing", "the invoice total does not match my plan"),
        ("hardware", "the battery drains within an hour of charging"),
        ("hardware", "the screen flickers when the brightness changes"),
        ("account", "I cannot reset my password from the email link"),
        ("account", "my login stopped working after the update"),
    ]
    with SOURCE.open("w", newline="") as handle:
        writer = csv.writer(handle)
        writer.writerow(["text", "label"])
        for index in range(20):
            label, text = ROWS[index % len(ROWS)]
            writer.writerow([f"{text} (ticket {index})", label])
    print("using the example instead:", SOURCE)

print(SOURCE.read_text().splitlines()[:3])

## 1 · Upload

In [ ]:
dataset = orinth.datasets.upload(SOURCE, name="text pipeline")
print(dataset.id, "·", dataset.task_type, "·", dataset.format)
print(dataset.labels, dataset.splits, "|", dataset.readiness.state)
orinth.datasets.load(dataset.id, "train", limit=4)

## 2 · Train

`source: "local"` options need no download and train in seconds; the
`hf_*` ones fetch a checkpoint the first time. Both are the same call.

In [ ]:
for entry in orinth.train.options("text_classification"):
    mark = "local " if entry["source"] == "local" else "download"
    print(f"  [{mark}] {entry['id']:34} {entry['family']}")

option = next(
    entry
    for entry in orinth.train.options("text_classification")
    if entry["runnable"] and not entry["needs_download"]
)
print("\ntraining with", option["id"])

In [ ]:
run = orinth.train.start(
    dataset.id,
    model_option_id=option["id"],
    task_type="text_classification",
    name="text pipeline",
    epochs=4,
    batch_size=8,
)
final = orinth.train.wait_for(run.id, on_progress=lambda r: print(f"{r.percent:5.1f}%  {r.step}"))
print("\n", final.status, "->", final.model_id, "|", final.error or "no error")

## 3 · Evaluate

In [ ]:
key = next(
    entry["key"]
    for entry in orinth.evaluate.datasets()
    if entry["key"].startswith(f"dataset:{dataset.id}:")
)
def overall_of(metrics):
    # Metrics are shaped per task — `image` for vision, `text_classification`
    # for text — so the headline block is *found* rather than guessed at by key.
    for value in metrics.values():
        if isinstance(value, dict) and isinstance(value.get("overall"), dict):
            return value["overall"]
    return {}


job = orinth.evaluate.start(final.model_id, key)
print(job.status, "|", job.error or "")
print({key_name: round(value, 3) for key_name, value in overall_of(job.metrics).items()})
print("labels:", job.metrics.get("labels"), "· samples:", job.metrics.get("samples"))

## 4 · Predict

`text=` instead of an image path. Same call, same recording — the answer lands
on `/inference` with an id.

In [ ]:
for sentence in [
    "my card was billed twice this month",
    "the display keeps flickering at low brightness",
]:
    answer = orinth.inference.predict(final.model_id, text=sentence)
    print(f"{answer.label:12} <- {sentence}")

In [ ]:
with orinth.runs.start(
    "text pipeline",
    params={"dataset": dataset.id, "model": final.model_id, "option": option["id"]},
    dataset_id=dataset.id,
) as pipeline:
    pipeline.log_text("labels", ", ".join(dataset.labels))
    pipeline.log(train_rows=float(dataset.splits.get("train", 0)))